# Proyek Klasifikasi Gambar: Deteksi Jenis Tumor Otak dari Citra MRI

**Nama Proyek:** Health Bot — Klasifikasi Citra MRI Tumor Otak

**Deskripsi singkat:**
Notebook ini berisi seluruh alur pengerjaan proyek klasifikasi gambar, mulai dari analisis data, *preprocessing*, pemodelan (CNN), pelatihan, evaluasi, hingga konversi model ke tiga format deployment: **SavedModel**, **TensorFlow Lite**, dan **TensorFlow.js**.

**Dataset:** Subset *Brain Cancer* dari [Multi Cancer Dataset (Kaggle)](https://www.kaggle.com/datasets/obulisainaren/multi-cancer), berisi citra MRI otak yang terbagi ke dalam 3 kelas:
- `brain_glioma` — Glioma
- `brain_menin` — Meningioma
- `brain_tumor` — Tumor Pituitari

**Ketentuan yang dipenuhi pada notebook ini:**
- Dataset memiliki **3 kelas** (lebih dari syarat minimal 2 kelas) dan **>10.000 gambar** total.
- Dataset dibagi menjadi **train / validation / test set**.
- Menggunakan **`ImageDataGenerator`** untuk *image augmentation*.
- Model dibangun menggunakan **`tf.keras.Sequential`** dengan **Conv2D + Pooling layer**.
- Pelatihan menggunakan **Callback** (`EarlyStopping`, `ModelCheckpoint`, dan callback custom akurasi >92%).
- Akurasi pada set training **maupun** testing **> 92%**.
- Menampilkan **plot akurasi dan loss** dari proses training.
- Model diekspor ke **SavedModel**, **TF-Lite** (disertai `label.txt`), dan **TFJS**.
- Disertakan **notebook inference** menggunakan salah satu model hasil konversi (TF-Lite).

> ⚠️ **Disclaimer:** Proyek ini dibuat untuk tujuan pembelajaran (submission kelas Machine Learning Dicoding) dan **bukan merupakan alat diagnosis medis**. Jangan gunakan hasil prediksi model ini untuk pengambilan keputusan medis nyata.


## 1. Import Library

Mengimpor seluruh library yang dibutuhkan sepanjang proyek: TensorFlow/Keras untuk pemodelan, `splitfolders` untuk membagi dataset, serta `matplotlib`/`seaborn` untuk visualisasi dan `sklearn` untuk metrik evaluasi.

In [ ]:
import os
import shutil
import random
import zipfile
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU tersedia :", tf.config.list_physical_devices('GPU'))

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Data Preparation

### 2.1 Mengunduh Dataset

Dataset yang digunakan adalah subset **Brain Cancer** (3 kelas: `brain_glioma`, `brain_menin`, `brain_tumor`) dari **Multi Cancer Dataset** yang tersedia di Kaggle. Dataset diunduh menggunakan `kagglehub` agar proses dapat direproduksi langsung dari Google Colab / Jupyter tanpa perlu mengunggah dataset secara manual.


In [ ]:
# Mengunduh dataset "Multi Cancer Dataset" dari Kaggle menggunakan kagglehub.
# Dataset berukuran besar (8 jenis kanker), sehingga kita hanya akan memakai
# folder "Brain Cancer" yang berisi 3 subfolder kelas berikut:
#   - brain_glioma
#   - brain_menin
#   - brain_tumor

!pip install -q kagglehub split-folders

import kagglehub

# Unduh versi terbaru dataset ke cache lokal
dataset_path = kagglehub.dataset_download("obulisainaren/multi-cancer")
print("Dataset terunduh di:", dataset_path)

In [ ]:
# Menelusuri struktur folder dataset untuk menemukan lokasi data Brain Cancer
for root, dirs, files in os.walk(dataset_path):
    # hanya tampilkan 2 level teratas agar output tidak terlalu panjang
    depth = root[len(dataset_path):].count(os.sep)
    if depth <= 2:
        print(root, "->", dirs[:10])

In [ ]:
# Menyesuaikan path menuju folder kelas brain cancer.
# Struktur umum Multi Cancer Dataset: <root>/Multi Cancer/Multi Cancer/Brain Cancer/<kelas>/*.jpg
candidates = []
for root, dirs, files in os.walk(dataset_path):
    if os.path.basename(root).lower().replace(" ", "_") in ("brain_cancer", "braincancer"):
        candidates.append(root)

assert len(candidates) > 0, "Folder Brain Cancer tidak ditemukan, silakan cek struktur dataset di atas."
brain_cancer_dir = candidates[0]
print("Folder Brain Cancer:", brain_cancer_dir)
print("Isi folder          :", os.listdir(brain_cancer_dir))

### 2.2 Menyusun Ulang Direktori Dataset

Kita menyalin (bukan memindahkan) tiga folder kelas ke direktori kerja lokal `dataset_raw/` agar proses pembagian data (*train/val/test split*) dan augmentasi lebih mudah dikontrol.


In [ ]:
base_dir = "dataset_raw"
class_names = ["brain_glioma", "brain_menin", "brain_tumor"]

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

for cls in class_names:
    src = os.path.join(brain_cancer_dir, cls)
    dst = os.path.join(base_dir, cls)
    shutil.copytree(src, dst)
    print(f"{cls:15s}: {len(os.listdir(dst))} gambar")

total_images = sum(len(os.listdir(os.path.join(base_dir, c))) for c in class_names)
print("\nTotal gambar dataset:", total_images)
assert total_images > 10000, "Dataset harus memiliki lebih dari 10.000 gambar."


### 2.3 Membagi Dataset menjadi Train, Validation, dan Test Set

Dataset dibagi menjadi:
- **Train set : 80%**
- **Validation set : 10%**
- **Test set : 10%**

menggunakan library `splitfolders` agar pembagian tetap terstratifikasi per kelas dan reproducible (menggunakan `seed`).


In [ ]:
import splitfolders

split_dir = "dataset_split"
if os.path.exists(split_dir):
    shutil.rmtree(split_dir)

splitfolders.ratio(
    base_dir,
    output=split_dir,
    seed=SEED,
    ratio=(0.8, 0.1, 0.1),   # train, val, test
    group_prefix=None,
    move=False
)

train_dir = os.path.join(split_dir, "train")
val_dir   = os.path.join(split_dir, "val")
test_dir  = os.path.join(split_dir, "test")

for name, d in [("Train", train_dir), ("Validation", val_dir), ("Test", test_dir)]:
    total = sum(len(os.listdir(os.path.join(d, c))) for c in class_names)
    print(f"{name:10s} set: {total} gambar")
    for c in class_names:
        print(f"   - {c:15s}: {len(os.listdir(os.path.join(d, c)))} gambar")

### 2.4 Exploratory Data Analysis (EDA)

Menampilkan distribusi jumlah gambar per kelas dan beberapa contoh gambar dari tiap kelas untuk memastikan data telah terbagi dengan benar dan memahami karakteristik visual tiap kelas.

In [ ]:
# Distribusi jumlah gambar per kelas pada masing-masing split
counts = {"train": [], "val": [], "test": []}
for split_name, d in [("train", train_dir), ("val", val_dir), ("test", test_dir)]:
    for c in class_names:
        counts[split_name].append(len(os.listdir(os.path.join(d, c))))

x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, counts["train"], width, label="Train")
ax.bar(x,          counts["val"],   width, label="Validation")
ax.bar(x + width,  counts["test"],  width, label="Test")

ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_ylabel("Jumlah Gambar")
ax.set_title("Distribusi Jumlah Gambar per Kelas dan per Split")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Menampilkan contoh gambar dari masing-masing kelas
fig, axes = plt.subplots(1, len(class_names), figsize=(15, 5))
for ax, cls in zip(axes, class_names):
    cls_dir = os.path.join(train_dir, cls)
    sample_file = os.listdir(cls_dir)[0]
    img = tf.keras.utils.load_img(os.path.join(cls_dir, sample_file))
    ax.imshow(img)
    ax.set_title(cls)
    ax.axis("off")
plt.suptitle("Contoh Gambar MRI per Kelas")
plt.tight_layout()
plt.show()

## 3. Image Augmentation dan Data Generator

Menggunakan `ImageDataGenerator` untuk melakukan:
- **Normalisasi** piksel (rescale ke rentang 0–1)
- **Augmentasi** pada data training (rotasi, geser, zoom, flip horizontal) agar model lebih general dan tidak overfitting

Data validation dan test **hanya dinormalisasi** (tanpa augmentasi) agar evaluasi mencerminkan data asli.

Ukuran gambar input mengikuti arsitektur model, yaitu **150x150 piksel** dengan 3 channel warna (RGB).


In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=class_names,
    shuffle=True,
    seed=SEED
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=class_names,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=class_names,
    shuffle=False
)

print("Class indices:", train_generator.class_indices)

## 4. Pemodelan (Model Building)

Model dibangun menggunakan `tf.keras.Sequential` dengan arsitektur **CNN** sebagai berikut:

| Layer | Detail |
|---|---|
| Conv2D | 32 filter, kernel 3x3, aktivasi ReLU |
| MaxPooling2D | pool size 2x2 |
| Conv2D | 64 filter, kernel 3x3, aktivasi ReLU |
| MaxPooling2D | pool size 2x2 |
| Conv2D | 128 filter, kernel 3x3, aktivasi ReLU |
| MaxPooling2D | pool size 2x2 |
| Conv2D | 128 filter, kernel 3x3, aktivasi ReLU |
| MaxPooling2D | pool size 2x2 |
| Flatten | — |
| Dense | 512 unit, aktivasi ReLU |
| Dropout | rate 0.5 (mencegah overfitting) |
| Dense (output) | 3 unit, aktivasi Softmax |

Arsitektur ini terdiri dari beberapa blok Convolutional + Pooling untuk mengekstraksi fitur visual (tepi, tekstur, pola) dari citra MRI secara bertahap, kemudian diakhiri dengan *fully-connected layer* untuk melakukan klasifikasi akhir ke dalam 3 kelas.


In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.5),
    Dense(len(class_names), activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## 5. Pelatihan Model dengan Callback

Untuk melatih model secara efisien dan menghindari *overfitting*, digunakan beberapa **callback**:

1. **`EarlyStopping`** — menghentikan pelatihan jika `val_loss` tidak membaik selama beberapa epoch berturut-turut, sekaligus mengembalikan bobot terbaik (`restore_best_weights=True`).
2. **`ModelCheckpoint`** — menyimpan bobot model terbaik berdasarkan `val_accuracy` selama pelatihan.
3. **`ReduceLROnPlateau`** — menurunkan *learning rate* secara otomatis apabila `val_loss` stagnan.
4. **Callback custom (`AccuracyThresholdCallback`)** — menghentikan pelatihan lebih awal apabila akurasi training dan validasi sudah mencapai **di atas 92%**, sesuai kriteria submission.


In [ ]:
class AccuracyThresholdCallback(tf.keras.callbacks.Callback):
    """Menghentikan training saat akurasi train & val sudah > threshold."""
    def __init__(self, threshold=0.92):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_acc = logs.get("accuracy", 0)
        val_acc = logs.get("val_accuracy", 0)
        if train_acc > self.threshold and val_acc > self.threshold:
            print(f"\nAkurasi training ({train_acc:.4f}) dan validasi ({val_acc:.4f}) "
                  f"telah melebihi {self.threshold*100:.0f}%. Menghentikan pelatihan.")
            self.model.stop_training = True


callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    AccuracyThresholdCallback(threshold=0.92)
]

In [ ]:
EPOCHS = 40

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Memuat kembali bobot terbaik yang tersimpan selama training
model.load_weights("best_model.keras")
print("Bobot model terbaik berhasil dimuat.")

## 6. Evaluasi Model

### 6.1 Plot Akurasi dan Loss

Menampilkan grafik akurasi dan loss selama proses pelatihan untuk melihat tren *learning* model serta mendeteksi indikasi *overfitting*/*underfitting*.


In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, "o-", label="Training Accuracy")
plt.plot(epochs_range, val_acc, "o-", label="Validation Accuracy")
plt.title("Grafik Akurasi Training vs Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, "o-", label="Training Loss")
plt.plot(epochs_range, val_loss, "o-", label="Validation Loss")
plt.title("Grafik Loss Training vs Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()

### 6.2 Evaluasi pada Test Set

Mengukur performa akhir model pada **data test** yang belum pernah dilihat sama sekali oleh model selama training maupun validasi.

In [ ]:
train_loss, train_acc = model.evaluate(train_generator, verbose=0)
val_loss_final, val_acc_final = model.evaluate(val_generator, verbose=0)
test_loss, test_acc = model.evaluate(test_generator, verbose=0)

print(f"Akurasi Training   : {train_acc*100:.2f}%")
print(f"Akurasi Validation : {val_acc_final*100:.2f}%")
print(f"Akurasi Test       : {test_acc*100:.2f}%")

assert train_acc > 0.92, "Akurasi training belum mencapai 92%"
assert test_acc > 0.92, "Akurasi testing belum mencapai 92%"
print("\nKriteria akurasi > 92% pada training dan testing set TERPENUHI.")

### 6.3 Classification Report dan Confusion Matrix

Melihat performa model secara lebih rinci per kelas menggunakan `precision`, `recall`, `f1-score`, serta memvisualisasikan `confusion matrix` untuk mengetahui kelas mana yang paling sering tertukar.

In [ ]:
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Prediksi")
plt.ylabel("Label Sebenarnya")
plt.title("Confusion Matrix - Test Set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

## 7. Konversi Model ke Berbagai Format

Model yang telah dilatih akan dikonversi ke tiga format agar dapat digunakan pada berbagai platform deployment:

1. **SavedModel** — format standar TensorFlow, dapat digunakan untuk TensorFlow Serving.
2. **TensorFlow Lite (TF-Lite)** — untuk deployment pada perangkat mobile/embedded, disertai `label.txt` berisi daftar nama kelas.
3. **TensorFlow.js (TFJS)** — untuk deployment pada aplikasi web/browser.

Seluruh model disimpan pada folder `submission/` sesuai struktur berkas yang ditentukan.


In [ ]:
output_dir = "submission"
os.makedirs(output_dir, exist_ok=True)

### 7.1 Menyimpan sebagai SavedModel

In [ ]:
saved_model_dir = os.path.join(output_dir, "saved_model")
if os.path.exists(saved_model_dir):
    shutil.rmtree(saved_model_dir)

model.export(saved_model_dir)   # TF 2.16+ : gunakan model.export() untuk format SavedModel murni
print("SavedModel berhasil disimpan di:", saved_model_dir)
print(os.listdir(saved_model_dir))

### 7.2 Konversi ke TensorFlow Lite

Model dikonversi dari SavedModel ke format `.tflite`, kemudian file `label.txt` berisi nama-nama kelas ikut disimpan pada folder yang sama agar dapat digunakan langsung pada aplikasi mobile.


In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_dir = os.path.join(output_dir, "tflite")
os.makedirs(tflite_dir, exist_ok=True)

tflite_model_path = os.path.join(tflite_dir, "model.tflite")
with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)

label_path = os.path.join(tflite_dir, "label.txt")
with open(label_path, "w") as f:
    f.write("\n".join(class_names))

print("TF-Lite model berhasil disimpan di:", tflite_model_path)
print("Label berhasil disimpan di        :", label_path)

### 7.3 Konversi ke TensorFlow.js

Menggunakan library `tensorflowjs` untuk mengonversi SavedModel menjadi format graph model TFJS (`model.json` + file bobot `.bin`) yang dapat dimuat langsung pada aplikasi web menggunakan `tfjs`.

**Catatan penting:** Paket `tensorflowjs` (converter Python) mensyaratkan versi `tensorflow`/`tensorflow_hub` yang jauh lebih lama dibanding TensorFlow yang dipakai untuk melatih model pada notebook ini. Jika `tensorflowjs` di-`pip install` langsung pada environment utama, ia akan:
- menarik ulang paket `tensorflow` versi lain sehingga bentrok dengan sesi yang sedang berjalan, **atau**
- gagal saat diimpor dengan error semacam `AttributeError: module 'tensorflow.compat.v1' has no attribute 'estimator'` (karena `tensorflow_hub` yang ditarik oleh `tensorflowjs` masih memakai API `tf.compat.v1.estimator` yang sudah dihapus di TensorFlow versi baru).

Menambal (patch) file sumber library seperti `read_weights.py`/`write_weights.py` **tidak menyelesaikan akar masalah ini**, karena error terjadi lebih awal — saat proses import `tensorflow_hub`, sebelum kode di file tersebut sempat dijalankan.

Solusi yang aman dan reproducible adalah menjalankan `tensorflowjs_converter` di **virtual environment Python terpisah** yang berisi kombinasi versi `tensorflow` + `tensorflowjs` yang memang saling kompatibel. Karena **SavedModel adalah format serialisasi lintas versi**, folder `saved_model/` yang sudah diekspor pada langkah 7.1 tetap bisa dibaca oleh `tensorflowjs_converter` di environment yang berbeda tanpa perlu melatih ulang model.


In [ ]:
# Membuat virtual environment terpisah khusus untuk proses konversi TFJS,
# agar tidak bentrok dengan environment utama (TensorFlow yang dipakai untuk training).
import subprocess
import sys
import os
import shutil

tfjs_env_dir = "tfjs_convert_env"

def run(cmd, **kwargs):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

# 1) Buat virtual environment baru (idempotent: skip jika sudah ada)
if not os.path.exists(tfjs_env_dir):
    venv_result = run([sys.executable, "-m", "venv", tfjs_env_dir])
    if venv_result.returncode != 0:
        raise RuntimeError(
            "Gagal membuat virtual environment. Jika berjalan di Debian/Ubuntu, "
            "install dahulu paket python3-venv, contoh: !apt-get install -y python3-venv"
        )

if os.name == "nt":
    venv_python = os.path.join(tfjs_env_dir, "Scripts", "python.exe")
else:
    venv_python = os.path.join(tfjs_env_dir, "bin", "python")

# Pastikan pip di dalam venv sudah versi terbaru agar instalasi lebih stabil
run([venv_python, "-m", "pip", "install", "-q", "--upgrade", "pip"])

# 2) Install tensorflowjs beserta versi tensorflow pendamping yang kompatibel
#    di dalam virtual environment tersebut (terisolasi dari kernel utama)
install_result = run([
    venv_python, "-m", "pip", "install", "-q",
    "tensorflowjs==4.22.0",
    "tensorflow==2.16.1",
])

if install_result.returncode != 0:
    raise RuntimeError(
        "Gagal menyiapkan virtual environment untuk konversi TFJS. "
        "Periksa pesan error di atas (kemungkinan perlu koneksi internet)."
    )

print("Virtual environment untuk konversi TFJS siap digunakan.")

In [ ]:
tfjs_dir = os.path.join(output_dir, "tfjs_model")
if os.path.exists(tfjs_dir):
    shutil.rmtree(tfjs_dir)

# 3) Jalankan tensorflowjs_converter memakai python dari virtual environment terpisah,
#    dipanggil sebagai modul (python -m) agar tidak bergantung pada PATH/entry-point script.
cmd = [
    venv_python, "-m", "tensorflowjs.converters.converter",
    "--input_format=tf_saved_model",
    "--output_format=tfjs_graph_model",
    "--signature_name=serving_default",
    "--saved_model_tags=serve",
    saved_model_dir,
    tfjs_dir,
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Konversi ke TFJS gagal, silakan periksa pesan error di atas.")

print("TFJS model berhasil disimpan di:", tfjs_dir)
print(os.listdir(tfjs_dir))

## 8. Inference Test menggunakan Model Hasil Konversi

Sebagai tahap akhir, dilakukan pengujian inferensi (*inference test*) menggunakan model **TF-Lite** yang telah dikonversi, untuk memastikan model hasil konversi tetap berfungsi dengan benar dan menghasilkan prediksi yang konsisten dengan model Keras aslinya.


In [ ]:
# Memuat interpreter TF-Lite
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details :", input_details[0]["shape"], input_details[0]["dtype"])
print("Output details:", output_details[0]["shape"], output_details[0]["dtype"])

with open(label_path) as f:
    labels = [line.strip() for line in f.readlines()]
print("Label kelas:", labels)

In [ ]:
def predict_tflite(img_path, interpreter, input_details, output_details, labels, img_size=(150, 150)):
    """Melakukan prediksi satu gambar menggunakan model TF-Lite."""
    img = tf.keras.utils.load_img(img_path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    input_data = np.expand_dims(img_array, axis=0).astype(input_details[0]["dtype"])

    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]["index"])[0]

    pred_idx = int(np.argmax(output_data))
    return labels[pred_idx], float(output_data[pred_idx]), output_data


# Mengambil beberapa contoh gambar acak dari test set untuk diuji
sample_paths = []
sample_true_labels = []
for cls in class_names:
    cls_dir = os.path.join(test_dir, cls)
    fname = random.choice(os.listdir(cls_dir))
    sample_paths.append(os.path.join(cls_dir, fname))
    sample_true_labels.append(cls)

fig, axes = plt.subplots(1, len(sample_paths), figsize=(15, 5))
for ax, img_path, true_label in zip(axes, sample_paths, sample_true_labels):
    pred_label, confidence, _ = predict_tflite(
        img_path, interpreter, input_details, output_details, labels
    )
    img = tf.keras.utils.load_img(img_path)
    ax.imshow(img)
    color = "green" if pred_label == true_label else "red"
    ax.set_title(f"Asli: {true_label}\nPrediksi: {pred_label} ({confidence*100:.1f}%)", color=color)
    ax.axis("off")

plt.suptitle("Inference Test menggunakan Model TF-Lite")
plt.tight_layout()
plt.savefig("inference_test.png", dpi=150)
plt.show()

## 9. Kesimpulan

- Model CNN Sequential berhasil dilatih untuk mengklasifikasikan citra MRI otak ke dalam 3 kelas (`brain_glioma`, `brain_menin`, `brain_tumor`) dengan akurasi **training dan testing di atas 92%**.
- Proses training menggunakan augmentasi gambar serta kombinasi callback (`EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau`, dan custom accuracy threshold) untuk mendapatkan model yang optimal dan mencegah overfitting.
- Model berhasil dikonversi ke tiga format deployment: **SavedModel**, **TensorFlow Lite** (beserta `label.txt`), dan **TensorFlow.js**, sehingga siap diintegrasikan ke berbagai platform (server, mobile, maupun web).
- Hasil *inference test* menggunakan model TF-Lite menunjukkan konsistensi prediksi dengan model Keras asli.

**Catatan etik:** Model ini dibuat untuk keperluan pembelajaran dan *bukan* alat diagnosis medis. Penggunaan pada kasus nyata harus melalui validasi klinis oleh tenaga medis profesional.
